# Shor factoring in OpenQARP: an exact small-integer reference

This notebook factors $N=15$ through OpenQARP's quantum order-finding circuit and classical Shor post-processing. The expected order is $\operatorname{ord}_{15}(2)=4$, producing the factor pair $(3,5)$.

> **Reference limitation:** modular multiplication is synthesized by enumerating basis permutations. It has exponential gate cost and a six-work-qubit limit, so this is a correctness reference for small integers—not a cryptographic-scale implementation or a speedup benchmark.

In [ ]:
import matplotlib.pyplot as plt

import qarp
from qarp.algorithms import Sampler
from qarp.algorithms import Shor
from qarp.blocks import OrderFindingBlock
from qarp.endianness import bits_to_label
from qarp.engines import QarpEngine

number = 15
base = 2
n_counting_qubits = 8

## Inspect the quantum order-finding result

The counting register is stored least-significant-bit first. With $t=8$ counting qubits, an outcome label $y$ represents the phase $y/2^8$. For order 4, the ideal peaks are therefore $y=0,64,128,192$.

In [ ]:
order_finding = OrderFindingBlock(
    base,
    number,
    n_counting_qubits=n_counting_qubits,
).build()
sampler = Sampler(
    ket=order_finding,
    n_shots=qarp.EXACT,
    measured_qubits=order_finding.counting_qubits,
)
engine = QarpEngine()
engine.build([sampler])
distribution = engine.run()[0]

peaks = sorted(
    (bits_to_label(bits), probability)
    for bits, probability in distribution.items()
    if probability > 1e-10
)
print("Nonzero phase bins:", peaks)
assert [label for label, _ in peaks] == [0, 64, 128, 192]
assert all(abs(probability - 0.25) < 1e-10 for _, probability in peaks)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(
    [label / 2**n_counting_qubits for label, _ in peaks],
    [probability for _, probability in peaks],
    width=0.02,
)
ax.set(xlabel="Measured phase", ylabel="Probability", title="Order finding for a=2 mod 15")
ax.set_xticks([0.0, 0.25, 0.5, 0.75])
ax.set_ylim(0.0, 0.3)
plt.show()

## Run the complete factoring workflow

Continued fractions recover and validate $r=4$. Since $2^{r/2}=4$ modulo 15, the classical stage evaluates $\gcd(4-1,15)=3$ and $\gcd(4+1,15)=5$. `force_quantum=True` guarantees the circuit runs: without it, Shor's per-attempt gcd shortcut may factor a small $N$ before any coprime base is tried, exactly as in the sequential textbook algorithm.

In [ ]:
shor = Shor(
    number,
    base=base,
    n_counting_qubits=n_counting_qubits,
    force_quantum=True,
    primitive=Sampler(n_shots=qarp.EXACT),
).build()
factors = shor.run()

print("Recovered order:", shor.orders[base])
print("Factors:", factors)
assert shor.orders[base] == 4
assert factors == (3, 5)

## Interpretation and sources

With finite shots, useful phase outcomes can be missed and `Shor.run()` may return `None`; configured attempts then move on to the next base in order. In the default mode (`force_quantum=False`) even inputs, exact perfect powers, and the first non-coprime base are handled classically in `run()`, and `Shor` accepts $N \le 64$ (six work qubits). The returned pair is not recursively decomposed into primes.

The workflow follows P. W. Shor, *Polynomial-Time Algorithms for Prime Factorization and Discrete Logarithms on a Quantum Computer*, SIAM J. Comput. 26, 1484–1509 (1997), [arXiv:quant-ph/9508027](https://arxiv.org/abs/quant-ph/9508027). Beauregard's [arXiv:quant-ph/0205095](https://arxiv.org/abs/quant-ph/0205095) is a design reference for a future scalable arithmetic backend.